# Notebook 2 — Extract Missing Domains from `yelp_sampled_100k.csv`

Extracts reviews for under-represented domains and appends them to `integrated_final_dataset.jsonl`,
following the exact schema used throughout the NaijaReview pipeline.

**Domains targeted:** `health`, `automotive`, `personal_care`, `entertainment`, `education`  
**Target per domain:** ~500 records (balanced, not bloating the dataset)  
**Output:** `integrated_final_dataset.jsonl` with new records appended + a domain audit log

> Run Notebook 1 first to confirm the CSV structure before running this.

In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 2: Config — edit these paths if your Drive layout differs ───────────
import os

BASE       = "/content/drive/MyDrive/NaijaReview_Data"
CSV_PATH   = f"{BASE}/yelp/yelp_sampled_100k.csv"
PROC_DIR   = f"{BASE}/Naiview/data/processed"
OUT_FILE   = f"{PROC_DIR}/integrated_final_dataset.jsonl"  # we APPEND to this
AUDIT_FILE = f"{PROC_DIR}/domain_extraction_audit.json"

# How many records to extract per domain (keep it balanced)
TARGET_PER_DOMAIN = 500

# Domains the existing dataset is thin on — confirmed by Notebook 1
MISSING_DOMAINS = ['health', 'automotive', 'personal_care', 'entertainment', 'education']

print(f"CSV:        {CSV_PATH}  exists={os.path.exists(CSV_PATH)}")
print(f"Output:     {OUT_FILE}  exists={os.path.exists(OUT_FILE)}")
print(f"Target:     {TARGET_PER_DOMAIN} records × {len(MISSING_DOMAINS)} domains = {TARGET_PER_DOMAIN * len(MISSING_DOMAINS):,} max new rows")

In [ ]:
# ── Cell 3: Domain keyword map (must match Notebook 1 & the pipeline) ────────

DOMAIN_KEYWORD_MAP = {
    'food':          ['restaurant', 'food', 'pizza', 'burger', 'sushi', 'bakery',
                      'breakfast', 'brunch', 'seafood', 'steakhouse', 'cafe',
                      'coffee', 'dessert', 'ice cream', 'juice', 'nigerian',
                      'african', 'mexican', 'italian', 'chinese', 'thai', 'deli',
                      'salad', 'chicken', 'sandwich', 'vegan', 'bbq'],
    'hospitality':   ['hotel', 'motel', 'hostel', 'resort', 'bed & breakfast',
                      'vacation rental', 'travel', 'airport', 'guest house'],
    'retail':        ['shopping', 'store', 'shop', 'market', 'bookstore', 'book',
                      'clothing', 'fashion', 'grocery', 'supermarket', 'convenience',
                      'furniture', 'electronics', 'jewel', 'gift', 'flower',
                      'liquor', 'beer', 'wine', 'spirits', 'home decor'],
    'services':      ['plumber', 'electrician', 'contractor', 'cleaning', 'repair',
                      'home service', 'laundry', 'dry clean', 'shipping', 'printing',
                      'notary', 'financial', 'insurance', 'legal', 'lawyer',
                      'accountant', 'tax', 'it service', 'computer', 'locksmith'],
    'health':        ['doctor', 'dentist', 'hospital', 'clinic', 'pharmacy',
                      'medical', 'optometrist', 'chiropractor', 'therapist',
                      'mental health', 'urgent care', 'pediatric', 'dermatolog',
                      'gynecolog', 'orthopedic', 'nursing', 'rehab'],
    'automotive':    ['auto', 'car', 'mechanic', 'tire', 'oil change', 'dealership',
                      'car wash', 'body shop', 'transmission', 'towing',
                      'motorcycle', 'truck', 'vehicle'],
    'personal_care': ['hair salon', 'barber', 'nail salon', 'spa', 'massage',
                      'beauty', 'waxing', 'skin care', 'facial', 'cosmetic',
                      'eyebrow', 'lash', 'tattoo', 'piercing'],
    'entertainment': ['nightlife', 'bar', 'club', 'karaoke', 'bowling', 'cinema',
                      'movie', 'theater', 'theatre', 'museum', 'gallery', 'art',
                      'arcade', 'escape room', 'comedy', 'concert', 'music',
                      'sport', 'fitness', 'gym', 'yoga', 'dance', 'park',
                      'recreation', 'amusement'],
    'education':     ['school', 'college', 'university', 'tutoring', 'education',
                      'training', 'driving school', 'language school', 'daycare',
                      'childcare', 'preschool', 'library'],
}

# Nigerian category name overrides (used in the 'category' field)
NIGERIAN_CATEGORY_NAMES = {
    'food':          'Food & Dining',
    'hospitality':   'Hotels & Travel',
    'retail':        'Shopping / Market',
    'services':      'Local Services',
    'health':        'Hospital / Clinic',
    'automotive':    'Mechanic / Auto',
    'personal_care': 'Salon & Spa',
    'entertainment': 'Entertainment / Nightlife',
    'education':     'School / Training',
    'general':       'General',
}

def infer_domain(text):
    """Infer domain from a raw category string. Returns the domain key."""
    if not isinstance(text, str):
        return 'general'
    t = text.lower()
    for domain, keywords in DOMAIN_KEYWORD_MAP.items():
        if any(kw in t for kw in keywords):
            return domain
    return 'general'

print("Domain map loaded. Domains to extract:", MISSING_DOMAINS)

In [ ]:
# ── Cell 4: Load CSV and identify category column ────────────────────────────
import pandas as pd

print("Loading CSV...")
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows. Columns: {list(df.columns)}")

# Identify which column holds Yelp category info
CATEGORY_COL = None
for c in ['categories', 'category', 'yelp_category', 'business_categories', 'domain']:
    if c in df.columns:
        CATEGORY_COL = c
        break

# Identify review text column
TEXT_COL = None
for c in ['text', 'review_text', 'review', 'body']:
    if c in df.columns:
        TEXT_COL = c
        break

# Identify star rating column
STARS_COL = None
for c in ['stars', 'rating', 'star_rating']:
    if c in df.columns:
        STARS_COL = c
        break

print(f"\nCategory col : {CATEGORY_COL}")
print(f"Text col     : {TEXT_COL}")
print(f"Stars col    : {STARS_COL}")

# Safety checks
assert TEXT_COL is not None,  "No review text column found. Check CSV columns above."
assert STARS_COL is not None, "No star rating column found. Check CSV columns above."

In [ ]:
# ── Cell 5: Derive domain for every row ──────────────────────────────────────

if CATEGORY_COL:
    df['_domain'] = df[CATEGORY_COL].apply(infer_domain)
elif 'domain' in df.columns:
    df['_domain'] = df['domain']
else:
    df['_domain'] = 'general'
    print("WARNING: No category column — all rows tagged 'general'.")

print("Current domain distribution in CSV:")
print(df['_domain'].value_counts().to_string())

In [ ]:
# ── Cell 6: Load existing integrated dataset to check what's already there ───
import json
from collections import Counter

existing_ids  = set()
existing_domains = Counter()

if os.path.exists(OUT_FILE):
    with open(OUT_FILE, 'r') as f:
        for line in f:
            try:
                rec = json.loads(line)
                existing_ids.add(rec.get('review_id', ''))
                existing_domains[rec.get('domain', 'unknown')] += 1
            except:
                pass
    print(f"Existing integrated dataset: {sum(existing_domains.values()):,} records")
    print("Domain breakdown:")
    for d, c in existing_domains.most_common():
        print(f"  {d:<20} {c:>6,}")
else:
    print("integrated_final_dataset.jsonl not found — will create it.")

In [ ]:
# ── Cell 7: Schema builder — maps CSV row → final_dataset record ─────────────
import uuid

def row_to_record(row, domain):
    """
    Converts a CSV row into a record matching the final_dataset.jsonl schema.
    Fields match exactly what the NaijaReview pipeline expects.
    """
    text     = str(row.get(TEXT_COL, '') or '').strip()
    stars    = float(row.get(STARS_COL, 3) or 3)
    cat_raw  = str(row.get(CATEGORY_COL, '') or '') if CATEGORY_COL else ''

    # Sentiment from star rating
    if stars >= 4:
        sentiment = 'positive'
        tier      = 'very_positive' if stars == 5 else 'positive'
    elif stars <= 2:
        sentiment = 'negative'
        tier      = 'very_negative' if stars == 1 else 'negative'
    else:
        sentiment = 'neutral'
        tier      = 'neutral'

    # Unique review ID — use existing one if present, otherwise generate
    review_id = str(row.get('review_id', '') or str(uuid.uuid4()))
    user_id   = str(row.get('user_id',   '') or 'unknown')
    item_id   = str(row.get('business_id', row.get('item_id', '')) or str(uuid.uuid4()))

    return {
        'review_id':             review_id,
        'user_id':               user_id,
        'item_id':               item_id,
        'text':                  text,
        'stars':                 stars,
        'sentiment':             sentiment,
        'tier':                  tier,
        'register':              'natural',
        'naija_mode':            False,
        'category':              NIGERIAN_CATEGORY_NAMES.get(domain, domain.title()),
        'yelp_category':         cat_raw[:120],  # cap length
        'domain':                domain,
        'category_confidence':   0.85,
        'region':                'Unknown',
        'pidgin_density':        0.0,
        'text_len':              len(text),
        'quality_score':         0.65,
        'nigerian_context_score': 0.0,
        'engagement':            int(row.get('useful', row.get('helpful', 0)) or 0),
        'date':                  str(row.get('date', '') or ''),
        'source':                'yelp_domain_expansion',
        'eval_text_quality':     True,
        'eval_rating_accuracy':  True,
        'eval_cross_domain':     True,
    }

print("Schema builder ready.")

In [ ]:
# ── Cell 8: Extract and write missing domains ────────────────────────────────
# This is the main extraction loop. For each missing domain we sample up to
# TARGET_PER_DOMAIN records, skip duplicates, and append to the output file.

import random
random.seed(42)

extraction_summary = {}
total_written = 0

with open(OUT_FILE, 'a') as out_f:
    for domain in MISSING_DOMAINS:
        already_have = existing_domains.get(domain, 0)
        need = max(0, TARGET_PER_DOMAIN - already_have)

        if need == 0:
            print(f"[{domain}] Already has {already_have:,} records — skipping.")
            extraction_summary[domain] = {'existing': already_have, 'extracted': 0}
            continue

        # Filter CSV for this domain
        domain_df = df[df['_domain'] == domain].copy()

        if len(domain_df) == 0:
            print(f"[{domain}] No rows found in CSV — domain not present in sampled data.")
            extraction_summary[domain] = {'existing': already_have, 'extracted': 0, 'note': 'not in CSV'}
            continue

        # Drop rows with empty text or very short reviews
        domain_df = domain_df[domain_df[TEXT_COL].astype(str).str.len() > 30]

        # Drop already-indexed review IDs
        if 'review_id' in domain_df.columns:
            domain_df = domain_df[~domain_df['review_id'].astype(str).isin(existing_ids)]

        # Sample
        sample_size = min(need, len(domain_df))
        sampled = domain_df.sample(n=sample_size, random_state=42)

        written = 0
        for _, row in sampled.iterrows():
            record = row_to_record(row.to_dict(), domain)
            if len(record['text']) < 30:   # final guard
                continue
            out_f.write(json.dumps(record) + '\n')
            existing_ids.add(record['review_id'])
            written += 1

        total_written += written
        extraction_summary[domain] = {
            'existing':   already_have,
            'extracted':  written,
            'available_in_csv': len(domain_df),
        }
        print(f"[{domain}] had {already_have:,}, extracted {written:,} new  "
              f"(CSV had {len(domain_df):,} candidates)")

print(f"\n✓ Total new records written: {total_written:,}")

In [ ]:
# ── Cell 9: Verify final domain distribution in the updated dataset ──────────
from collections import Counter

final_domains = Counter()
total_records = 0

with open(OUT_FILE, 'r') as f:
    for line in f:
        try:
            rec = json.loads(line)
            final_domains[rec.get('domain', 'unknown')] += 1
            total_records += 1
        except:
            pass

print(f"{'='*45}")
print(f"FINAL integrated_final_dataset.jsonl SUMMARY")
print(f"{'='*45}")
print(f"Total records: {total_records:,}")
print(f"\n{'Domain':<22} {'Count':>8}  {'Share':>7}")
print("-" * 45)
for domain, count in final_domains.most_common():
    share = count / total_records * 100
    print(f"{domain:<22} {count:>8,}  {share:>6.1f}%")

# Save audit
audit = {
    'extraction_summary': extraction_summary,
    'final_domain_distribution': dict(final_domains),
    'total_records': total_records,
}
with open(AUDIT_FILE, 'w') as f:
    json.dump(audit, f, indent=2)
print(f"\nAudit saved to: {AUDIT_FILE}")

In [ ]:
# ── Cell 10: Visualise before vs after ───────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

domains   = sorted(final_domains.keys())
before    = [existing_domains.get(d, 0) for d in domains]
after     = [final_domains.get(d, 0) for d in domains]
extracted = [after[i] - before[i] for i in range(len(domains))]

x   = np.arange(len(domains))
w   = 0.35
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(x - w/2, before, w, label='Before extraction', color='#5B8DB8', edgecolor='black')
ax.bar(x + w/2, after,  w, label='After extraction',  color='#E8873A', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(domains, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Record count')
ax.set_title('Domain Coverage: Before vs After Extraction', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/content/domain_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to /content/domain_before_after.png")